In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
BIN_PATH = '/Users/alizad/PycharmProjects/NewWorking-MUBES'

sys.path.insert(0, str(BIN_PATH))


import os,sys


import geopandas as gpd
import pickle
#import pickle5
import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
from bin.outputs import output_utilities
import core.GeneralFunctions as GrlFct
import building_geometry.BuildingObject as BuildingObject
import pandas as pd

import outputs.output_utilities as Utilities
import building_geometry.BuildingObject as BuildingObject

import json

import os


In [2]:
import pickle
import sys
import types
import numpy as np

ListofOri = []

path = os.getcwd()[:os.getcwd().find(('NewWorking-MUBES'))+16]+'/examples/1965-1975/Final_Sim_Results/SvB_Base'
files = os.listdir(path)

# ---- only needed because your pickle uses MUBES custom class ----
mod = types.ModuleType("building_geometry")
sub = types.ModuleType("building_geometry.BuildingObject")

class Building:
    pass

sub.Building = Building
mod.BuildingObject = sub
sys.modules["building_geometry"] = mod
sys.modules["building_geometry.BuildingObject"] = sub
# ---------------------------------------------------------------

for pckl in files:
    if pckl.endswith('.pickle'):
        pickle_path = os.path.join(path, pckl)

        with open(pickle_path, "rb") as f:
            data = pickle.load(f)

        bd = data["BuildData"]

        UUID = bd.BuildID['50A_UUID']
        # footprint from MUBES object
        footprint = np.array(bd.footprint[0], dtype=float)

        def azimuth_from_vector(dx, dy):
            return (np.degrees(np.arctan2(dx, dy)) + 360) % 360

        # ---- Method 1: longest edge orientation ----
        edges = []



        for i in range(len(footprint)):
            p1 = footprint[i]
            p2 = footprint[(i + 1) % len(footprint)]

            dx, dy = p2 - p1
            length = np.sqrt(dx**2 + dy**2)
            azimuth = azimuth_from_vector(dx, dy)

            edges.append({
                "edge": i + 1,
                "length": length,
                "azimuth": azimuth
            })

        longest_edge = max(edges, key=lambda x: x["length"])

        print("Longest edge method")
        print("Building axis:", round(longest_edge["azimuth"], 2), "degrees from North")
        print("Opposite direction:", round((longest_edge["azimuth"] + 180) % 360, 2))

        # ---- Method 2: PCA orientation ----
        centered = footprint - footprint.mean(axis=0)

        cov = np.cov(centered.T)
        eigvals, eigvecs = np.linalg.eigh(cov)

        main_vector = eigvecs[:, np.argmax(eigvals)]

        pca_azimuth = azimuth_from_vector(main_vector[0], main_vector[1])

        # because building axis has no head, normalize to 0–180
        pca_axis = pca_azimuth % 180

        ListofOri.append({'ID': UUID, 'Orientation': pca_axis})
        print("\nPCA method")
        print("Building axis:", round(pca_axis, 2), "degrees from North")
        print("Opposite direction:", round((pca_axis + 180) % 360, 2))

Longest edge method
Building axis: 222.35 degrees from North
Opposite direction: 42.35

PCA method
Building axis: 42.36 degrees from North
Opposite direction: 222.36
Longest edge method
Building axis: 111.13 degrees from North
Opposite direction: 291.13

PCA method
Building axis: 111.19 degrees from North
Opposite direction: 291.19
Longest edge method
Building axis: 315.72 degrees from North
Opposite direction: 135.72

PCA method
Building axis: 135.74 degrees from North
Opposite direction: 315.74
Longest edge method
Building axis: 204.93 degrees from North
Opposite direction: 24.93

PCA method
Building axis: 21.27 degrees from North
Opposite direction: 201.27
Longest edge method
Building axis: 357.07 degrees from North
Opposite direction: 177.07

PCA method
Building axis: 177.07 degrees from North
Opposite direction: 357.07
Longest edge method
Building axis: 176.83 degrees from North
Opposite direction: 356.83

PCA method
Building axis: 176.86 degrees from North
Opposite direction: 356

In [ ]:
path = os.getcwd()[:os.getcwd().find(('NewWorking-MUBES'))+16]+'/examples/1965-1975/Final_Sim_Results'
files = os.listdir(path)
results_by_building = {}

for SvBECM in files:
    path = os.getcwd()[:os.getcwd().find(('NewWorking-MUBES'))+16]+'/examples/1965-1975/Final_Sim_Results'
    if any(keyword in SvBECM.lower() for keyword in ['wall', 'window', 'roof', 'base', 'heatrecovery','wall_window', 'wall_roof','wall_heatrecovery','window_roof', 'window_heatrecovery','roof_heatrecovery','wall_window_roof','facade_roof_heatrecovery', 'wall_window_heatrecovery', 'window_roof_heatrecovery', 'wall_window_roof_heatrecovery']):

        path = os.path.join(path, SvBECM, 'Sim_Results')

        # Initialize inner dict for this ECM
        results_by_building[SvBECM] = {}

        ResultFiles = os.listdir(path)

        for bld in ResultFiles:
            if bld.endswith('.pickle'):
                with open(os.path.join(path, bld), 'rb') as f:
                    results_by_building[SvBECM][bld] = pickle.load(f)
assert isinstance(results_by_building, dict)


In [4]:
print(sum(results_by_building['SvB_Base']['Building_138v0.pickle']['HeatedArea']['Data_Zone Ideal Loads Supply Air Total Heating Rate'])/4244158)

print((sum(results_by_building['SvB_Base']['Building_138v0.pickle']['HeatedArea']['Data_Zone Ideal Loads Supply Air Total Heating Energy'])/3600000)/4244.158)


61.39393012292694
61.39393012292701


In [5]:
extraVar = ['nbAppartments', 'ATempOr', 'SharedBld', 'height', 'StoreyHeigth', 'nbfloor','BlocHeight','BlocFootprintArea','BlocNbFloor', 'HeatedArea', 'AreaBasedFlowRate','NonHeatedArea', 'Other']
SumResult = {}
MainPath = os.getcwd()[:os.getcwd().find(('NewWorking-MUBES'))+16]+'/examples/1965-1975/Final_Sim_Results'

files = os.listdir(MainPath)
for SvBECM in files:
    if any(keyword in SvBECM.lower() for keyword in ['base_withvent', 'wall', 'window', 'roof', 'base', 'heatrecovery','wall_window', 'wall_roof','wall_heatrecovery','window_roof', 'window_heatrecovery','roof_heatrecovery','wall_window_roof','wall_roof_heatrecovery', 'wall_window_heatrecovery', 'window_roof_heatrecovery', 'wall_window_roof_heatrecovery']):
        path = os.path.join(MainPath, SvBECM, 'Sim_Results')
        # reuse the pickles already loaded in `results_by_building` (cell above) instead of
        # re-reading ~2700 files from disk; GetData output is byte-for-byte identical.
        SumResult[SvBECM] = Utilities.GetData(path, extraVar, PreLoadedResults=results_by_building.get(SvBECM))

reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...
reading file...
organizing data...


In [ ]:
path2CP = '/Users/alizad/OneDrive - KTH/infrastructure/FME/DataProducts/Output/CPBuildings/KTH_CityPlanner_Stockholm'
FinalResult = {}
cpBuildings = pd.read_csv(os.path.join(path2CP, 'cpBuildings.csv'), sep=';', encoding='utf-8')
cpProperties = pd.read_csv(os.path.join(path2CP, 'cpProperties.csv'), sep=';', encoding='utf-8')
# SvBWithUUID = pd.read_excel('/Users/alizad/OneDrive - KTH/Kallvatten_byggnad_with_addresses_and_ids_and_BldgUUD.xlsx' )
cpNow = pd.read_csv(os.path.join(path2CP, 'cpNow.csv'), sep=';', encoding='utf-8')
cpNow['FormularId'] = cpNow['FormularId'].astype(int)
cpFootprint = gpd.read_file(os.path.join(path2CP, 'cpFootprints.geojson'))
for SvBECM in SumResult.keys():
    AllBuldings = []
    Res = SumResult[SvBECM]
    for idx in range(len(Res['HeatedArea'])):
        Azimuth = [az['Orientation'] for az in ListofOri if az['ID']==Res['BuildID'][idx]['50A_UUID']]
        try:
            FormularID = int(cpFootprint.loc[cpFootprint["50A_UUID"] == Res['BuildID'][idx]['50A_UUID'], "FormularId"].item())
            FNR = cpFootprint.loc[cpFootprint["50A_UUID"] == Res['BuildID'][idx]['50A_UUID'], "FNR"].item()
            # Address = cpProperties.loc[cpProperties['FNR'].astype(str).str.lstrip('0') == str(FNR).lstrip('0'),'42P_FAL_UTADR2'].iloc[0]
            # Address = SvBWithUUID.loc[SvBWithUUID["55A_UUIDREGBYG"] == Res['BuildID'][idx]['50A_UUID'], "Building"].item()
            Declaration = cpNow.loc[cpNow['FormularId']==FormularID, 'EgiEnergiklass'].item()
            AntalPlan = cpNow.loc[cpNow['FormularId']==FormularID, 'EgenAntalPlan'].item()
            AntalTrapphus = cpNow.loc[cpNow['FormularId']==FormularID, 'EgenAntalTrapphus'].item()
            BuildingType = cpBuildings.loc[cpBuildings['50A_UUID']==Res['BuildID'][idx]['50A_UUID'], '50A_ByggnadsKategori'].item()
            EPC_SpaceHeating =  cpNow.loc[cpNow['FormularId']==FormularID, 'EgiUppvarmning_calc'].item()
            if np.isnan(PrimEnergy):
                PrimEnergy = 0

        except:
            FormularID = '00000000'
            Declaration = '0'
            PrimEnergy = 0
        AllBuldings.append({'ID': Res['BuildID'][idx]['50A_UUID'],
                            'EP_Area': Res['DB_Surf'][idx],
                            'Calc_Area': Res['EP_Area'][idx],
                            'Azimuth':Azimuth[0],
                            'HeatDemandNoVent':SumResult['SvB_Base']['EP_Heat'][idx],
                            'HeatDemandWithVent':SumResult['SvB_Base_withvent']['EP_Heat'][idx],
                            'HeatDemandAfterECM':SumResult[SvBECM]['EP_Heat'][idx],
                            'EPC_Corrected': Res['EPC_Heat'][idx]*Res['DB_Surf'][idx]/Res['EP_Area'][idx],
                            'EPC_Record': Res['EPC_Heat'][idx],
                            'SpaceHeating': sum(Res['HeatedArea'][idx]['Data_Zone Ideal Loads Supply Air Total Heating Rate'])/1000/Res['EP_Area'][idx],
                            'DHW': Res['EPC_DHW'][idx],#['Data_Water Use Equipment Heating Rate'])/(1000*Res['EP_Area'][idx]),
                            'OnSiteGen': sum(Res['Other'][idx]['Data_Electric Load Center Produced Electricity Energy'])/3600000/Res['EP_Area'][idx],
                            'SurplusElec': sum(Res['Other'][idx]['Data_ElectricitySurplusSold:Facility'])/3600000/Res['EP_Area'][idx],
                            'WindowLoss':sum(Res['HeatedArea'][idx]['Data_Zone Windows Total Heat Loss Energy'])/3600000/Res['EP_Area'][idx],
                            'WallLoss': sum(Res['HeatedArea'][idx]['Data_Surface Inside Face Conduction Heat Loss Rate On Vertical Walls'])/1000/Res['EP_Area'][idx],
                            'RoofLoss':sum(Res['HeatedArea'][idx]['Data_Surface Inside Face Conduction Heat Loss Rate On Roofs'])/1000/Res['EP_Area'][idx],
                            'HeatRecovery': sum(Res['HeatedArea'][idx]['Data_Zone Ideal Loads Heat Recovery Total Heating Energy'])/3600000/Res['EP_Area'][idx],
                            'Incident solar': sum(Res['HeatedArea'][idx]['Data_Surface Outside Face Incident Solar Radiation Rate per Area On Vertical Walls'])/1000,
                            'PrimEnergy': PrimEnergy,
                            'Declaration': Declaration,
                            'BuildingType': BuildingType,
                            'nbFloor': AntalPlan,
                            'nbStairCase': AntalTrapphus,
                            })
    if SvBECM == 'SvB_Base':
        continue
    else:
        FinalResult[SvBECM] = AllBuldings
FinalResult["SvB_Base"] = FinalResult.pop("SvB_Base_withvent")
results_by_building['SvB_Base']['Building_279v0.pickle']['BuildDB'].EPCMeters['Heating']['DistrictHeating']



In [8]:
import copy
base = FinalResult['SvB_Base']
DeltaResult = copy.deepcopy(FinalResult)
for sav in FinalResult:
    if sav == 'SvB_Base':
        continue
    else:
        for bld in range(len(base)):
            if FinalResult[sav][bld]['ID'] == base[bld]['ID']:
                for meas in FinalResult[sav][bld]:
                    if any(key in meas for key in ['ID', 'BuildingType', 'Azimuth','Address', 'EP_Area', 'OnSiteGen', 'SurplusElec', 'Incident solar', 'Declaration', 'PrimEnergy', 'nbFloor', 'nbStairCase', 'HeatDemand', 'EPC_Demand', 'Calc_Area', 'EPC_Record', 'EPC_Corrected', 'EPC_New']):
                        continue
                    else:
                        if meas == 'BaseHeatDemand':
                            continue
                        else:
                            DeltaResult[sav][bld][meas] = FinalResult[sav][bld][meas] - base[bld][meas]
                            if meas == 'HeatRecovery'and DeltaResult[sav][bld][meas] != 0:
                                DeltaResult[sav][bld][meas] = - DeltaResult[sav][bld][meas]


KeyboardInterrupt: 

In [10]:
import json
with open(os.getcwd()[:os.getcwd().find(('NewWorking-MUBES'))+16]+'/examples/1965-1975/Analysis_Temp/Dynamic_SvB.json', 'w') as f:
    json.dump(DeltaResult, f, indent=4)

In [11]:
import json, os
import pandas as pd

dynamic_path = os.getcwd()[
                   :os.getcwd().find('NewWorking-MUBES') + 16] + '/examples/1965-1975/Analysis_Temp/Dynamic_SvB.json'

with open(dynamic_path, 'r') as f:
    DeltaResult = json.load(f)

csv = pd.DataFrame(DeltaResult['SvB_Base'])

csv.to_csv(os.getcwd()[
                   :os.getcwd().find('NewWorking-MUBES') + 16] + '/examples/1965-1975/Analysis_Temp/Dynamic.csv', index=False)


In [11]:
for i in FinalResult['SvB_Base']:
    idfied = 0
    for idx, j in enumerate(FinalResult['SvB_Wall_Roof']):
        if i['ID'] == j['ID']:
            idfied = 1
            break
        else:
            if idx == len(FinalResult['SvB_Wall_Roof'])-1 and idfied == 0:
                print(i['ID'])
                print(idx)



3e2ff899-a390-4739-b529-843abc8b08c5
154
026ec25b-ecec-426f-bbb8-4e53e1b864a1
154
ad9a2363-b936-4364-afa6-8f8e0b8131bb
154


In [10]:
for IDX, i in enumerate(FinalResult['SvB_Base']):
    idfied = 0
    for idx, j in enumerate(FinalResult['SvB_Wall_Roof']):
        if i['ID'] == j['ID']:
            idfied = 1
            break
        else:
            if idx == len(FinalResult['SvB_Wall_Roof'])-1 and idfied == 0:
                print(i['ID'])
                print(IDX)

3e2ff899-a390-4739-b529-843abc8b08c5
4
026ec25b-ecec-426f-bbb8-4e53e1b864a1
99
ad9a2363-b936-4364-afa6-8f8e0b8131bb
115
